# Funciones auxiliares

Carga de datos y evaluación con SMOTE solo para train.


In [ ]:
from utils import load_data, evaluate_multiple_models, get_features_root



# Extracción de features 

En el ZIP ya están guardados los resultados (140mins de ejecución aprox)


In [ ]:
import os
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from tsfresh import extract_features
from tsfresh.feature_extraction import EfficientFCParameters, ComprehensiveFCParameters
from imblearn.over_sampling import SMOTE


devices = ["emotibit_volar"]
overlap_modes = [
    "sin_solapamiento",
    "con_solapamiento",
]

stamps_dir = os.path.join("data", "Stamps")
TSFRESH_N_JOBS = 1
USER_N_JOBS = max(1, (os.cpu_count() or 1) - 1)
WINDOW_OVERLAP_FRACTION = 0.5  # 50% de solape temporal entre ventanas


# AJUSTAR CLASES SI ES NECESARIO
segments = [
    ("relax_start", "relax_end", "Baseline", "Relax"),
    ("squat_test_start", "squat_test_end", "Squat test", "Squat"),
    ("video1_start", "video1_end", "Video 1", "Disgusting"),
    ("video2_start", "video2_end", "Video 2", "Sadness"),
]


def extract_window(df, start, end):
    return df[(df["Timestamp"] >= start) & (df["Timestamp"] < end)]


def process_user(device, user, window, window_step, stamps_dir, segments):
    user_path = os.path.join("data", device, user)

    if not os.path.isdir(user_path):
        return []

    gsr_path = os.path.join(user_path, "Gsr.csv")
    hr_path = os.path.join(user_path, "Hr.csv")
    stamp_path = os.path.join(stamps_dir, f"{user}.txt")

    if not all(map(os.path.exists, [gsr_path, hr_path, stamp_path])):
        return []

    print(f"Procesando {device} - {user}...")

    gsr_df = pd.read_csv(gsr_path)
    hr_df = pd.read_csv(hr_path)

    stamps = {}
    with open(stamp_path) as f:
        for line in f:
            try:
                k, v = line.strip().split(",")
                stamps[k] = float(v)
            except ValueError:
                continue

    baseline_gsr = extract_window(
        gsr_df, stamps["relax_start"], stamps["relax_end"]
    )["Gsr"]
    if not baseline_gsr.empty:
        mu_gsr = baseline_gsr.mean()
        sigma_gsr = baseline_gsr.std()
        if sigma_gsr > 0:
            gsr_df["Gsr"] = (gsr_df["Gsr"] - mu_gsr) / sigma_gsr
        else:
            gsr_df["Gsr"] = gsr_df["Gsr"] - mu_gsr

    baseline_hr = extract_window(
        hr_df, stamps["relax_start"], stamps["relax_end"]
    )["Hr"]
    if not baseline_hr.empty:
        mu_hr = baseline_hr.mean()
        sigma_hr = baseline_hr.std()
        if sigma_hr > 0:
            hr_df["Hr"] = (hr_df["Hr"] - mu_hr) / sigma_hr
        else:
            hr_df["Hr"] = hr_df["Hr"] - mu_hr

    user_rows = []

    for start_k, end_k, task, label in segments:
        if start_k not in stamps or end_k not in stamps:
            continue

        limit_start = stamps[start_k]
        limit_end = stamps[end_k]

        window_starts = np.arange(limit_start, limit_end, window_step)
        gsr_windows = []
        hr_windows = []

        for win_start in window_starts:
            win_end = win_start + window
            if win_end > limit_end:
                continue

            gsr_win = extract_window(gsr_df, win_start, win_end)
            hr_win = extract_window(hr_df, win_start, win_end)
            if gsr_win.empty or hr_win.empty:
                continue

            gsr_windows.append(
                gsr_win.rename(columns={"Gsr": "value"}).assign(
                    id=len(gsr_windows),
                    time=gsr_win["Timestamp"],
                    kind="Gsr",
                )
            )

            hr_windows.append(
                hr_win.rename(columns={"Hr": "value"}).assign(
                    id=len(hr_windows),
                    time=hr_win["Timestamp"],
                    kind="Hr",
                )
            )

        if not gsr_windows or not hr_windows:
            continue

        gsr_concat = pd.concat(gsr_windows, axis=0)
        hr_concat = pd.concat(hr_windows, axis=0)

        gsr_features = extract_features(
            gsr_concat,
            column_id="id",
            column_sort="time",
            column_kind="kind",
            column_value="value",
            default_fc_parameters=EfficientFCParameters(),
            n_jobs=TSFRESH_N_JOBS,
            disable_progressbar=True,
        )

        hr_features = extract_features(
            hr_concat,
            column_id="id",
            column_sort="time",
            column_kind="kind",
            column_value="value",
            default_fc_parameters=EfficientFCParameters(),
            n_jobs=TSFRESH_N_JOBS,
            disable_progressbar=True,
        )

        for idx in gsr_features.index:
            row = {"Class": label, "User": int(user)}
            row.update(gsr_features.loc[idx].to_dict())
            row.update(hr_features.loc[idx].to_dict())
            user_rows.append(row)

    return user_rows


for overlap_mode in overlap_modes:

    print(f"\n================================================")
    print(f"Modo de ventanas: {overlap_mode}")
    print(f"================================================")

    # sin_solapamiento: paso = WINDOW
    # con_solapamiento: paso menor que WINDOW, por defecto WINDOW/2

    for device in devices:

        print(f"\n############################################")
        print(f"Procesando dispositivo: {device}")
        print(f"############################################")

        output_root = get_features_root(overlap_mode=overlap_mode)
        output_dir = os.path.join(output_root, f"features_extracted_{device}")
        os.makedirs(output_dir, exist_ok=True)
        device_path = os.path.join("data", device)
        users = sorted(
            user for user in os.listdir(device_path)
            if os.path.isdir(os.path.join(device_path, user))
        )

        for WINDOW in range(1, 16):

            print(f"\n==============================")
            print(f"Procesando WINDOW = {WINDOW} segundos")
            print(f"==============================")

            user_results = Parallel(n_jobs=min(USER_N_JOBS, len(users)))(
                delayed(process_user)(
                    device,
                    user,
                    WINDOW,
                    WINDOW * (1 - WINDOW_OVERLAP_FRACTION) if overlap_mode == "con_solapamiento" else WINDOW,
                    stamps_dir,
                    segments,
                )
                for user in users
            )

            dataset_rows = [row for user_rows in user_results for row in user_rows]

            df = pd.DataFrame(dataset_rows)
            if df.empty:
                continue

            df = df.dropna(axis=1)
            meta_cols = ["Class", "User"]
            numeric_cols = df.drop(columns=meta_cols, errors="ignore").select_dtypes(include=np.number)
            numeric_cols = numeric_cols.loc[:, numeric_cols.var() > 0]
            df_final = pd.concat([df[meta_cols], numeric_cols], axis=1)

            df_final.to_csv(
                os.path.join(output_dir, f"features_window_{WINDOW}.csv"),
                index=False,
            )

            print(
                f"Features extraidas para {device} WINDOW={WINDOW} guardadas en {output_dir}"
            )



# Evaluar cada ventana

Usa las funciones auxiliar del principio (con SMOTE).



In [ ]:
from utils import load_data, evaluate_multiple_models, get_features_root



In [ ]:
import os
import json
import logging
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


WINDOW_EVAL_N_JOBS = max(1, (os.cpu_count() or 1) - 1)
overlap_modes = [
    "sin_solapamiento",
    "con_solapamiento",
]


def to_builtin(obj):
    if isinstance(obj, dict):
        return {str(k): to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_builtin(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_builtin(v) for v in obj]
    if hasattr(obj, "item"):
        return obj.item()
    return obj


def summarize_window_flags(serializable_results):
    models_with_incident = []
    flag_counts = {}

    for model_name, model_results in serializable_results.items():
        flags = model_results.get("flags", {})
        if flags.get("has_incident"):
            models_with_incident.append(model_name)
        for flag_name, count in flags.get("flag_counts", {}).items():
            flag_counts[flag_name] = flag_counts.get(flag_name, 0) + count

    return {
        "has_incident": len(models_with_incident) > 0,
        "models_with_incident": models_with_incident,
        "flag_counts": flag_counts,
    }


def evaluate_window(device, window, smote_option, overlap_mode):
    models = {
        "RandomForest": RandomForestClassifier(
            n_estimators=500,
            n_jobs=1,
            random_state=42,
        ),
        "KNN": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=5)),
        ]),
        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, random_state=42)),
        ]),
    }

    X, y, groups = load_data(device, window, overlap_mode=overlap_mode)

    results = evaluate_multiple_models(
        models,
        X,
        y,
        groups,
        n_splits=10,
        features=X.columns.tolist(),
        smote=smote_option,
        return_flags=True,
    )

    serializable_results = {}
    for model_name, metrics in results.items():
        serializable_results[model_name] = {
            "accuracy": {
                "folds": [float(x) for x in metrics["accuracy"]["folds"]],
                "mean": float(metrics["accuracy"]["mean"]),
                "std": float(metrics["accuracy"]["std"]),
            },
            "f1_mean": {
                "folds": [float(x) for x in metrics["f1_mean"]["folds"]],
                "mean": float(metrics["f1_mean"]["mean"]),
                "std": float(metrics["f1_mean"]["std"]),
            },
            "f1_macro": {
                "folds": [float(x) for x in metrics["f1_macro"]["folds"]],
                "mean": float(metrics["f1_macro"]["mean"]),
                "std": float(metrics["f1_macro"]["std"]),
            },
            "flags": to_builtin(metrics.get("_flags", {})),
        }

    return window, {
        "_window_flags": summarize_window_flags(serializable_results),
        **serializable_results,
    }


devices = ["emotibit_volar"]

smote_options = [
    False,
    "random",
    "smote",
    "smoteenn",
    "smotetomek"
]

for overlap_mode in overlap_modes:

    print(f"\n================================================")
    print(f"Evaluando modo de ventanas: {overlap_mode}")
    print(f"================================================")

    for device in devices:

        print(f"\n############################################")
        print(f"Evaluando dispositivo: {device}")
        print(f"############################################")

        features_folder = os.path.join(
            get_features_root(overlap_mode=overlap_mode),
            f"features_extracted_{device}",
        )

        for smote_option in smote_options:

            print(f"\n============================================")
            print(f"{device} | {overlap_mode} -> SMOTE = {smote_option}")
            print(f"============================================")

            window_results = Parallel(n_jobs=WINDOW_EVAL_N_JOBS)(
                delayed(evaluate_window)(device, window, smote_option, overlap_mode)
                for window in range(1, 16)
            )

            all_results = {}
            for window, window_payload in sorted(window_results, key=lambda x: x[0]):
                print(f"\n\n==============================")
                print(f"{device} | {overlap_mode} -> SMOTE = {smote_option} -> WINDOW = {window}")
                print(f"==============================")

                for model_name, metrics in window_payload.items():
                    if model_name.startswith("_"):
                        continue

                    print(f"\n{model_name}")
                    for metric_name in ["accuracy", "f1_mean", "f1_macro"]:
                        metric_values = metrics.get(metric_name, {})
                        mean = metric_values.get("mean")
                        std = metric_values.get("std")
                        print(f"{metric_name}: {mean:.4f} +/- {std:.4f}")

                    flags = metrics.get("flags", {})
                    if flags.get("has_incident"):
                        print(
                            "flags:",
                            flags.get("flag_counts", {}),
                            "| folds:",
                            flags.get("folds_with_incident", []),
                        )
                    else:
                        print("flags: OK")

                all_results[f"window_{window}"] = window_payload

            smote_name = str(smote_option)
            results_path = os.path.join(
                features_folder,
                f"results_all_windows_{smote_name}.json",
            )

            with open(results_path, "w") as f:
                json.dump(to_builtin(all_results), f, indent=4)

            print(f"\nResultados completos guardados en {results_path}")



# Gráficas


In [ ]:
from utils import get_features_root

import os
import json
import numpy as np
import matplotlib.pyplot as plt

devices = ["emotibit_volar"]
overlap_modes = [
    "sin_solapamiento",
    "con_solapamiento",
]

smote_methods = [
    False,

    "random",
    "smote",
    "smoteenn",
    "smotetomek",
]

metric = "f1_mean"
max_window = 10


def annotate_peaks(ax, windows, values, color):
    values = np.asarray(values)
    windows = np.asarray(windows)

    if len(values) < 3:
        peak_idx = np.arange(len(values))
    else:
        peak_idx = []
        for i in range(len(values)):
            left_ok = (i == 0) or (values[i] > values[i - 1])
            right_ok = (i == len(values) - 1) or (values[i] > values[i + 1])
            if left_ok and right_ok:
                peak_idx.append(i)
        peak_idx = np.array(peak_idx, dtype=int)

    for i in peak_idx:
        ax.annotate(
            f"{values[i]:.3f}",
            (windows[i], values[i]),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=8,
            color=color,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.7),
        )


for overlap_mode in overlap_modes:
    for smote_method in smote_methods:

        print(
            f"\nGraficando resultados side by side | overlap_mode={overlap_mode} | SMOTE: {smote_method}"
        )

        fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

        for ax, device in zip(axes, devices):

            features_folder = os.path.join(
                get_features_root(overlap_mode=overlap_mode),
                f"features_extracted_{device}",
            )
            json_path = os.path.join(
                features_folder,
                f"results_all_windows_{smote_method}.json",
            )

            if not os.path.exists(json_path):
                print(f"No se encontro {json_path}")
                ax.set_title(f"{device}\nJSON no encontrado")
                ax.axis("off")
                continue

            with open(json_path, "r") as f:
                results = json.load(f)

            all_models = set()

            for window in range(1, max_window + 1):
                key = f"window_{window}"

                if key not in results:
                    continue

                all_models.update(
                    model_name
                    for model_name in results[key].keys()
                    if not model_name.startswith("_")
                )

            all_models = sorted(all_models)

            for model_name in all_models:
                windows = []
                metric_mean = []
                metric_std = []

                for window in range(1, max_window + 1):
                    key = f"window_{window}"

                    if key not in results:
                        continue
                    if model_name not in results[key]:
                        continue

                    model_metrics = results[key][model_name]

                    if metric not in model_metrics:
                        continue
                    if "mean" not in model_metrics[metric]:
                        continue
                    if model_metrics[metric].get("mean") is None:
                        continue

                    windows.append(window)
                    metric_mean.append(model_metrics[metric]["mean"])
                    metric_std.append(model_metrics[metric].get("std", 0))

                if len(windows) == 0:
                    continue

                windows = np.array(windows)
                metric_mean = np.array(metric_mean)
                metric_std = np.array(metric_std)

                line = ax.errorbar(
                    windows,
                    metric_mean,
                    yerr=metric_std,
                    marker="o",
                    capsize=4,
                    linewidth=1.8,
                    label=model_name,
                )

                color = line.lines[0].get_color()
                annotate_peaks(ax, windows, metric_mean, color)

            ax.grid(True, which="both", linestyle="--", linewidth=0.6, alpha=0.7)
            ax.set_xticks(range(1, max_window + 1))
            ax.set_ylim(0.25, 1.05)
            ax.set_xlabel("Ventana (s)")
            ax.set_title(f"{device} | {overlap_mode}")

        axes[0].set_ylabel("F1 weighted")
        axes[1].legend(loc="lower right", fontsize=9, frameon=True)

        plt.tight_layout()
        plt.show()



In [ ]:
device = "emotibit_volar"
smote_method = False

overlap_modes_to_export = [
    "sin_solapamiento",
    "con_solapamiento",
]

output_dir = "figuras_pdf"
os.makedirs(output_dir, exist_ok=True)

for overlap_mode in overlap_modes_to_export:
    
    print(
        f"\nExportando figura individual | device={device} | "
        f"overlap_mode={overlap_mode} | SMOTE: {smote_method}"
    )

    fig, ax = plt.subplots(1, 1, figsize=(7, 5))

    features_folder = os.path.join(
        get_features_root(overlap_mode=overlap_mode),
        f"features_extracted_{device}",
    )

    json_path = os.path.join(
        features_folder,
        f"results_all_windows_{smote_method}.json",
    )

    if not os.path.exists(json_path):
        print(f"No se encontro {json_path}")
        plt.close(fig)
        continue

    with open(json_path, "r") as f:
        results = json.load(f)

    all_models = set()

    for window in range(1, max_window + 1):
        key = f"window_{window}"

        if key not in results:
            continue

        all_models.update(
            model_name
            for model_name in results[key].keys()
            if not model_name.startswith("_")
        )

    all_models = sorted(all_models)

    for model_name in all_models:
        windows = []
        metric_mean = []
        metric_std = []

        for window in range(1, max_window + 1):
            key = f"window_{window}"

            if key not in results:
                continue
            if model_name not in results[key]:
                continue

            model_metrics = results[key][model_name]

            if metric not in model_metrics:
                continue
            if "mean" not in model_metrics[metric]:
                continue
            if model_metrics[metric].get("mean") is None:
                continue

            windows.append(window)
            metric_mean.append(model_metrics[metric]["mean"])
            metric_std.append(model_metrics[metric].get("std", 0))

        if len(windows) == 0:
            continue

        windows = np.array(windows)
        metric_mean = np.array(metric_mean)
        metric_std = np.array(metric_std)

        line = ax.errorbar(
            windows,
            metric_mean,
            yerr=metric_std,
            marker="o",
            capsize=4,
            linewidth=1.8,
            label=model_name,
        )

        color = line.lines[0].get_color()
        annotate_peaks(ax, windows, metric_mean, color)

    ax.grid(True, which="both", linestyle="--", linewidth=0.6, alpha=0.7)
    ax.set_xticks(range(1, max_window + 1))
    ax.set_ylim(0.45, 1.05)
    ax.set_xlabel("Ventana (s)")
    ax.set_ylabel("F1 weighted")

    # if overlap_mode == "con_solapamiento":
    #     ax.set_title(f"Ventanas temporales con solapamiento")
    # else:
    #     ax.set_title(f"Ventanas temporales sin solapamiento")
    
    ax.legend(loc="lower right", fontsize=9, frameon=True)

    plt.tight_layout()

    pdf_name = f"{device}_{overlap_mode}_SMOTE_{smote_method}.pdf"
    pdf_path = os.path.join(output_dir, pdf_name)

    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")

    print(f"Figura guardada en: {pdf_path}")

    plt.show()
    plt.close(fig)


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

devices = ["emotibit_volar"]
overlap_modes = [
    "sin_solapamiento",
    "con_solapamiento",
]

smote_methods = [
    False,
    "random",
    "smote",
    "smoteenn",
    "smotetomek",
]

smote_labels = {
    False: "False",
    "random": "ROS",
    "smote": "SMOTE",
    "smoteenn": "SMOTEENN",
    "smotetomek": "SMOTETomek",
}

metric = "f1_mean"
model_name = "RandomForest"
max_window = 15


def annotate_peaks(ax, windows, values, color):
    values = np.asarray(values)
    windows = np.asarray(windows)

    if len(values) < 3:
        peak_idx = np.arange(len(values))
    else:
        peak_idx = []
        for i in range(len(values)):
            left_ok = (i == 0) or (values[i] > values[i - 1])
            right_ok = (i == len(values) - 1) or (values[i] > values[i + 1])
            if left_ok and right_ok:
                peak_idx.append(i)
        peak_idx = np.array(peak_idx, dtype=int)

    for i in peak_idx:
        continue
        ax.annotate(
            f"{values[i]:.3f}",
            (windows[i], values[i]),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=8,
            color=color,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.7),
        )


for overlap_mode in overlap_modes:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    for ax, device in zip(axes, devices):

        for smote_method in smote_methods:

            features_folder = os.path.join(
                get_features_root(overlap_mode=overlap_mode),
                f"features_extracted_{device}",
            )
            json_path = os.path.join(
                features_folder,
                f"results_all_windows_{smote_method}.json",
            )

            if not os.path.exists(json_path):
                print(f"No se encontro {json_path}")
                continue

            with open(json_path, "r") as f:
                results = json.load(f)

            windows = []
            metric_mean = []
            metric_std = []

            for window in range(1, max_window + 1):
                key = f"window_{window}"

                if key not in results:
                    continue
                if model_name not in results[key]:
                    continue

                model_metrics = results[key][model_name]

                if metric not in model_metrics:
                    continue
                if "mean" not in model_metrics[metric]:
                    continue
                if model_metrics[metric].get("mean") is None:
                    continue

                windows.append(window)
                metric_mean.append(model_metrics[metric]["mean"])
                metric_std.append(model_metrics[metric].get("std", 0))

            if len(windows) == 0:
                continue

            windows = np.array(windows)
            metric_mean = np.array(metric_mean)
            metric_std = np.array(metric_std)

            line = ax.errorbar(
                windows,
                metric_mean,
                yerr=metric_std,
                marker="o",
                capsize=4,
                linewidth=1.8,
                label=smote_labels[smote_method],
            )

            color = line.lines[0].get_color()
            annotate_peaks(ax, windows, metric_mean, color)

        ax.grid(True, which="both", linestyle="--", linewidth=0.6, alpha=0.7)
        ax.set_xticks(range(1, max_window + 1))
        ax.set_ylim(0.25, 1.05)
        ax.set_xlabel("Ventana (s)")
        ax.set_title(f"{device} | {model_name} | {overlap_mode}")

    axes[0].set_ylabel("F1 weighted")
    axes[1].legend(loc="lower right", fontsize=9, frameon=True)

    plt.tight_layout()
    plt.show()



In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

device = "emotibit_volar"

overlap_modes = [
    "sin_solapamiento",
    "con_solapamiento",
]

smote_methods = [
    False,
    "random",
    "smote",
    "smoteenn",
    "smotetomek",
]

smote_labels = {
    False: "False",
    "random": "ROS",
    "smote": "SMOTE",
    "smoteenn": "SMOTEENN",
    "smotetomek": "SMOTETomek",
}

metric = "f1_mean"
model_name = "RandomForest"
max_window = 10

output_folder = "figuras_pdf"
os.makedirs(output_folder, exist_ok=True)


def annotate_peaks(ax, windows, values, color):
    values = np.asarray(values)
    windows = np.asarray(windows)

    if len(values) < 3:
        peak_idx = np.arange(len(values))
    else:
        peak_idx = []
        for i in range(len(values)):
            left_ok = (i == 0) or (values[i] > values[i - 1])
            right_ok = (i == len(values) - 1) or (values[i] > values[i + 1])
            if left_ok and right_ok:
                peak_idx.append(i)
        peak_idx = np.array(peak_idx, dtype=int)

    for i in peak_idx:
        continue
        ax.annotate(
            f"{values[i]:.3f}",
            (windows[i], values[i]),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=8,
            color=color,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.7),
        )


for overlap_mode in overlap_modes:

    fig, ax = plt.subplots(figsize=(7, 5))

    for smote_method in smote_methods:

        features_folder = os.path.join(
            get_features_root(overlap_mode=overlap_mode),
            f"features_extracted_{device}",
        )

        json_path = os.path.join(
            features_folder,
            f"results_all_windows_{smote_method}.json",
        )

        if not os.path.exists(json_path):
            print(f"No se encontró {json_path}")
            continue

        with open(json_path, "r") as f:
            results = json.load(f)

        windows = []
        metric_mean = []
        metric_std = []

        for window in range(1, max_window + 1):
            key = f"window_{window}"

            if key not in results:
                continue
            if model_name not in results[key]:
                continue

            model_metrics = results[key][model_name]

            if metric not in model_metrics:
                continue
            if "mean" not in model_metrics[metric]:
                continue
            if model_metrics[metric].get("mean") is None:
                continue

            windows.append(window)
            metric_mean.append(model_metrics[metric]["mean"])
            metric_std.append(model_metrics[metric].get("std", 0))

        if len(windows) == 0:
            continue

        windows = np.array(windows)
        metric_mean = np.array(metric_mean)
        metric_std = np.array(metric_std)

        line = ax.errorbar(
            windows,
            metric_mean,
            yerr=metric_std,
            marker="o",
            capsize=4,
            linewidth=1.8,
            label=smote_labels[smote_method],
        )

        color = line.lines[0].get_color()
        annotate_peaks(ax, windows, metric_mean, color)

    ax.grid(True, which="both", linestyle="--", linewidth=0.6, alpha=0.7)
    ax.set_xticks(range(1, max_window + 1))
    ax.set_ylim(0.25, 1.05)
    ax.set_xlabel("Ventana (s)")
    ax.set_ylabel("F1 weighted")
    # ax.set_title(f"{device} | {model_name} | {overlap_mode}")
    ax.legend(loc="lower right", fontsize=9, frameon=True)

    plt.tight_layout()

    pdf_path = os.path.join(
        output_folder,
        f"{device}_{model_name}_{overlap_mode}.pdf"
    )

    plt.savefig(pdf_path, format="pdf", bbox_inches="tight")
    print(f"Figura guardada en: {pdf_path}")

    plt.show()
    plt.close(fig)


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# Si no lo tienes ya importado arriba:
# from utils import get_features_root

device = "emotibit_volar"

overlap_modes = [
    "sin_solapamiento",
    "con_solapamiento",
]

smote_methods = [
    False,
    "random",
    "smote",
    "smoteenn",
    "smotetomek",
]

smote_labels = {
    False: "Original",
    "random": "ROS",
    "smote": "SMOTE",
    "smoteenn": "SMOTEENN",
    "smotetomek": "SMOTETomek",
}

metric = "f1_mean"
model_name = "RandomForest"
max_window = 10

output_folder = "figuras_pdf"
os.makedirs(output_folder, exist_ok=True)


def get_peak_indices(values):
    values = np.asarray(values)

    if len(values) < 3:
        return np.arange(len(values))

    peak_idx = []

    for i in range(len(values)):
        left_ok = (i == 0) or (values[i] > values[i - 1])
        right_ok = (i == len(values) - 1) or (values[i] > values[i + 1])

        if left_ok and right_ok:
            peak_idx.append(i)

    return np.array(peak_idx, dtype=int)


def annotate_best_peaks_by_window(ax, series_data):
    """
    Etiqueta máximos locales, pero si varios métodos tienen un máximo
    en la misma ventana, solo etiqueta el método con mayor valor.
    """

    best_peak_by_window = {}

    for item in series_data:
        windows = item["windows"]
        values = item["values"]
        color = item["color"]
        label = item["label"]

        peak_idx = get_peak_indices(values)

        for i in peak_idx:
            window = int(windows[i])
            value = float(values[i])

            if window not in best_peak_by_window:
                best_peak_by_window[window] = {
                    "value": value,
                    "color": color,
                    "label": label,
                    "window": window,
                }
            else:
                if value > best_peak_by_window[window]["value"]:
                    best_peak_by_window[window] = {
                        "value": value,
                        "color": color,
                        "label": label,
                        "window": window,
                    }

    for window, peak in best_peak_by_window.items():
        ax.annotate(
            # f"{peak['label']}\n{peak['value']:.3f}",
            f"{peak['value']:.3f}",
            (peak["window"], peak["value"]),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=8,
            color=peak["color"],
            bbox=dict(
                boxstyle="round,pad=0.2",
                fc="white",
                ec=peak["color"],
                alpha=0.75,
            ),
        )


for overlap_mode in overlap_modes:

    fig, ax = plt.subplots(figsize=(7, 5))

    series_data = []

    for smote_method in smote_methods:

        features_folder = os.path.join(
            get_features_root(overlap_mode=overlap_mode),
            f"features_extracted_{device}",
        )

        json_path = os.path.join(
            features_folder,
            f"results_all_windows_{smote_method}.json",
        )

        if not os.path.exists(json_path):
            print(f"No se encontró {json_path}")
            continue

        with open(json_path, "r") as f:
            results = json.load(f)

        windows = []
        metric_mean = []
        metric_std = []

        for window in range(1, max_window + 1):
            key = f"window_{window}"

            if key not in results:
                continue
            if model_name not in results[key]:
                continue

            model_metrics = results[key][model_name]

            if metric not in model_metrics:
                continue
            if "mean" not in model_metrics[metric]:
                continue
            if model_metrics[metric].get("mean") is None:
                continue

            windows.append(window)
            metric_mean.append(model_metrics[metric]["mean"])
            metric_std.append(model_metrics[metric].get("std", 0))

        if len(windows) == 0:
            continue

        windows = np.array(windows)
        metric_mean = np.array(metric_mean)
        metric_std = np.array(metric_std)

        line = ax.errorbar(
            windows,
            metric_mean,
            yerr=metric_std,
            marker="o",
            capsize=4,
            linewidth=1.8,
            label=smote_labels[smote_method],
        )

        color = line.lines[0].get_color()

        series_data.append(
            {
                "windows": windows,
                "values": metric_mean,
                "color": color,
                "label": smote_labels[smote_method],
            }
        )

    annotate_best_peaks_by_window(ax, series_data)

    ax.grid(True, which="both", linestyle="--", linewidth=0.6, alpha=0.7)
    ax.set_xticks(range(1, max_window + 1))
    ax.set_ylim(0.45, 1.05)
    ax.set_xlabel("Ventana (s)")
    ax.set_ylabel("F1 weighted")
    # ax.set_title(f"{device} | {model_name} | {overlap_mode}")
    ax.legend(loc="lower right", fontsize=9, frameon=True)

    plt.tight_layout()

    pdf_path = os.path.join(
        output_folder,
        f"{device}_{model_name}_{overlap_mode}.pdf"
    )

    plt.savefig(pdf_path, format="pdf", bbox_inches="tight")
    print(f"Figura guardada en: {pdf_path}")

    plt.show()
    plt.close(fig)


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

devices = ["emotibit_volar"]
overlap_modes = [
    "sin_solapamiento",
    "con_solapamiento",
]

smote_methods = [
    False,
    "random",
    "smote",
    "smoteenn",
    "smotetomek",
]

smote_labels = {
    False: "False",
    "random": "ROS",
    "smote": "SMOTE",
    "smoteenn": "SMOTEENN",
    "smotetomek": "SMOTETomek",
}

metric = "f1_mean"
model_name = "RandomForest"
max_window = 15


def annotate_peaks(ax, windows, values, color):
    values = np.asarray(values)
    windows = np.asarray(windows)

    if len(values) < 3:
        peak_idx = np.arange(len(values))
    else:
        peak_idx = []
        for i in range(len(values)):
            left_ok = (i == 0) or (values[i] > values[i - 1])
            right_ok = (i == len(values) - 1) or (values[i] > values[i + 1])
            if left_ok and right_ok:
                peak_idx.append(i)
        peak_idx = np.array(peak_idx, dtype=int)

    for i in peak_idx:
        continue
        ax.annotate(
            f"{values[i]:.3f}",
            (windows[i], values[i]),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=8,
            color=color,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.7),
        )


for overlap_mode in overlap_modes:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    for ax, device in zip(axes, devices):

        for smote_method in smote_methods:

            features_folder = os.path.join(
                get_features_root(overlap_mode=overlap_mode),
                f"features_extracted_{device}",
            )
            json_path = os.path.join(
                features_folder,
                f"results_all_windows_{smote_method}.json",
            )

            if not os.path.exists(json_path):
                print(f"No se encontro {json_path}")
                continue

            with open(json_path, "r") as f:
                results = json.load(f)

            windows = []
            metric_mean = []
            metric_std = []

            for window in range(1, max_window + 1):
                key = f"window_{window}"

                if key not in results:
                    continue
                if model_name not in results[key]:
                    continue

                model_metrics = results[key][model_name]

                if metric not in model_metrics:
                    continue
                if "mean" not in model_metrics[metric]:
                    continue
                if model_metrics[metric].get("mean") is None:
                    continue

                windows.append(window)
                metric_mean.append(model_metrics[metric]["mean"])
                metric_std.append(model_metrics[metric].get("std", 0))

            if len(windows) == 0:
                continue

            windows = np.array(windows)
            metric_mean = np.array(metric_mean)
            metric_std = np.array(metric_std)

            line = ax.errorbar(
                windows,
                metric_mean,
                yerr=metric_std,
                marker="o",
                capsize=4,
                linewidth=1.8,
                label=smote_labels[smote_method],
            )

            color = line.lines[0].get_color()
            annotate_peaks(ax, windows, metric_mean, color)

        ax.grid(True, which="both", linestyle="--", linewidth=0.6, alpha=0.7)
        ax.set_xticks(range(1, max_window + 1))
        ax.set_ylim(0.25, 1.05)
        ax.set_xlabel("Ventana (s)")
        ax.set_title(f"{device} | {model_name} | {overlap_mode}")

    axes[0].set_ylabel("F1 weighted")
    axes[1].legend(loc="lower right", fontsize=9, frameon=True)

    plt.tight_layout()
    plt.show()



In [ ]:
import os
import json

devices = [
    "emotibit_volar",
]
overlap_modes = [
    "sin_solapamiento",
    "con_solapamiento",
]

smote_methods = [False, "random", "smote", "smoteenn", "smotetomek"]
max_window = 15

for overlap_mode in overlap_modes:
    for smote_method in smote_methods:
        print(f"\nOverlap mode: {overlap_mode} | Smote method: {smote_method}")

        for device in devices:
            features_folder = os.path.join(
                get_features_root(overlap_mode=overlap_mode),
                f"features_extracted_{device}",
            )
            json_path = os.path.join(
                features_folder,
                f"results_all_windows_{smote_method}.json",
            )

            if not os.path.exists(json_path):
                print(f"No se encontro {json_path}")
                continue

            with open(json_path, "r") as f:
                results = json.load(f)

            best_by_model = {}

            for window in range(1, max_window + 1):
                key = f"window_{window}"

                if key not in results:
                    continue

                for model_name, model_results in results[key].items():
                    if model_name.startswith("_"):
                        continue
                    if "f1_mean" not in model_results:
                        continue

                    f1_mean = model_results["f1_mean"].get("mean")
                    f1_std = model_results["f1_mean"].get("std")

                    if f1_mean is None:
                        continue

                    if model_name not in best_by_model:
                        best_by_model[model_name] = {
                            "best_window": window,
                            "best_f1_mean": f1_mean,
                            "best_f1_std": f1_std,
                            "flags": model_results.get("flags", {}),
                        }
                    elif f1_mean > best_by_model[model_name]["best_f1_mean"]:
                        best_by_model[model_name] = {
                            "best_window": window,
                            "best_f1_mean": f1_mean,
                            "best_f1_std": f1_std,
                            "flags": model_results.get("flags", {}),
                        }

            print(f"\nDevice: {device}")

            if not best_by_model:
                print("No se encontraron resultados validos.")
                continue

            for model_name, info in best_by_model.items():
                std_text = (
                    f"{info['best_f1_std']:.4f}"
                    if info["best_f1_std"] is not None
                    else "N/A"
                )
                flags = info.get("flags", {})

                print(
                    f"  {model_name}: mejor ventana = {info['best_window']} segundos "
                    f"con F1-weighted = {info['best_f1_mean']:.4f} +/- {std_text}"
                )

                if flags.get("has_incident"):
                    print(
                        f"    incidencias: folds={flags.get('folds_with_incident', [])} "
                        f"| flags={flags.get('flag_counts', {})}"
                    )
                else:
                    print("    incidencias: OK")

